# Unified Virmen + Kilosort Data Conversion to NWB

This notebook demonstrates how to convert Tank lab Virmen behavioral data and Kilosort spike sorting data to NWB format with synchronized timestamps from the U19 DataJoint database.

## Key Features:
- Database queries are performed in the notebook, not in data interfaces
- Metadata and synchronized timestamps are passed as external parameters
- The converter and interfaces remain database-agnostic and reusable
- Supports optional data modalities (Kilosort, Suite2p, etc.)

## 1. Import Required Libraries

In [1]:
import sys
from pathlib import Path
from datetime import datetime
import numpy as np

# Add U19 pipeline to path for DataJoint access
sys.path.insert(0, '/Users/ct5868/code/U19-pipeline_python')

# Import Tank lab converter and interfaces
# Import for reading NWB files
from pynwb import NWBHDF5IO

from tank_lab_to_nwb.convert_towers_task.towersnwbconverter import TowersNWBConverter
from tank_lab_to_nwb.convert_towers_task.virmenbehaviordatainterface import VirmenDataInterface

print("All imports successful!")

All imports successful!


## 2. Setup DataJoint Connection (Optional)

This section configures the connection to the U19 DataJoint database. You can skip this if you don't have database access - the converter will still work with data from the Virmen file alone.

In [ ]:
# Import DataJoint and U19 pipeline modules
try:
    # from scripts.conf_file_finding import try_find_conf_file
    # try_find_conf_file()

    import datajoint as dj
    import u19_pipeline.ephys_pipeline as ep
    import u19_pipeline.acquisition as acquisition

    # Create virtual modules for querying
    subject = dj.create_virtual_module("subject", "u19_subject")
    lab = dj.create_virtual_module("lab", "u19_lab")
    recording = dj.create_virtual_module("recording", "u19_recording")

    DB_AVAILABLE = True
    print("DataJoint connection successful!")
except Exception as e:
    print(f"Error importing DataJoint or U19 pipeline modules: {e}")
    DB_AVAILABLE = False
    print(f"DataJoint not available: {e}")
    print("Proceeding without database access - will use only Virmen file data")

## 3. Load Virmen Behavioral Data

Load the reference Jorge Virmen file and create the VirmenDataInterface.

In [ ]:
# Define file paths for actual test data
virmen_file_path = Path("/Users/ct5868/code/testing_yanar/jorge_pwmv2_cohort1_185A-Rig1_jyanar_ya008_T_20240525_0.mat")

if not virmen_file_path.exists():
    raise FileNotFoundError(f"Virmen file not found: {virmen_file_path}")

# Create VirmenDataInterface
virmen_interface = VirmenDataInterface(file_path=virmen_file_path, verbose=True)

# Kilosort data paths (two probes, using higher job IDs)
kilosort_base = Path("/Users/ct5868/code/testing_yanar/known_kilosort_structure/ya008_20240525_g0")
kilosort_probe0_path = kilosort_base / "ya008_20240525_g0_imec0" / "job_id_680" / "kilosort4_output"
kilosort_probe1_path = kilosort_base / "ya008_20240525_g0_imec1" / "job_id_681" / "kilosort4_output"

# Verify files exist
print(f"Virmen file exists: {virmen_file_path.exists()}")
print(f"Kilosort probe 0 exists: {kilosort_probe0_path.exists()}")
print(f"Kilosort probe 1 exists: {kilosort_probe1_path.exists()}")
print(f"\nVirmen file: {virmen_file_path}")
print(f"Kilosort probe 0: {kilosort_probe0_path}")
print(f"Kilosort probe 1: {kilosort_probe1_path}")



Virmen file exists: True
Kilosort probe 0 exists: True
Kilosort probe 1 exists: True

Virmen file: /Users/ct5868/code/testing_yanar/jorge_pwmv2_cohort1_185A-Rig1_jyanar_ya008_T_20240525_0.mat
Kilosort probe 0: /Users/ct5868/code/testing_yanar/known_kilosort_structure/ya008_20240525_g0/ya008_20240525_g0_imec0/job_id_680/kilosort4_output
Kilosort probe 1: /Users/ct5868/code/testing_yanar/known_kilosort_structure/ya008_20240525_g0/ya008_20240525_g0_imec1/job_id_681/kilosort4_output


## 4. Extract Session Key Information

Use the `get_session_key()` method to extract subject and session date from the Virmen file. This information is used to query the database for recording ID and synchronized timestamps.

In [ ]:
# Define session to query - ya008, session on 2024-05-25
session_key = {
    'subject_fullname': 'jyanar_ya008',
    'session_date': '2024-05-25',
}

print(f"Querying session: {session_key}")

Querying session: {'subject_fullname': 'jyanar_ya008', 'session_date': '2024-05-25'}


## 5. Query Metadata and Sync Timestamps from Database

Query DataJoint for:
1. Experimenter information (owner and co-owners)  
2. Recording ID  
3. Pre-computed synchronized timestamps from BehaviorSync table

In [ ]:
# Initialize variables
experimenter_list = []
recording_id = None
sync_timestamps = None
subject_sex = None
subject_date_of_birth = None

if DB_AVAILABLE:
    try:
        # Query subject and experimenter information
        subject_fullname = session_key['subject_fullname']
        sub_info = (subject.Subject() * lab.User() & f"subject_fullname = '{subject_fullname}'").fetch1()

        # Get owner user_id and fetch full name
        owner_id = sub_info["user_id"]
        owner_info = (lab.User() & f"user_id = '{owner_id}'").fetch1()
        owner_full_name = owner_info.get("full_name", owner_id)

        # Format owner name as "LastName, FirstName" (DANDI compliant)
        if owner_full_name and ' ' in owner_full_name:
            name_parts = owner_full_name.rsplit(' ', 1)  # Split by last space
            owner_formatted = f"{name_parts[-1]}, {name_parts[0]}"  # "LastName, FirstName"
        else:
            owner_formatted = owner_full_name

        # Get coowners and format their names
        coowner_ids = list(
            (subject.SubjectCoowners() & f"subject_fullname = '{subject_fullname}' and active = 1").fetch("coowner")
        )

        coowners_formatted = []
        for coowner_id in coowner_ids:
            coowner_info = (lab.User() & f"user_id = '{coowner_id}'").fetch1()
            coowner_full_name = coowner_info.get("full_name", coowner_id)

            # Format coowner name as "LastName, FirstName"
            if coowner_full_name and ' ' in coowner_full_name:
                name_parts = coowner_full_name.rsplit(' ', 1)
                coowner_formatted = f"{name_parts[-1]}, {name_parts[0]}"
            else:
                coowner_formatted = coowner_full_name

            coowners_formatted.append(coowner_formatted)

        experimenter_list = [owner_formatted] + coowners_formatted
        print(f"Experimenters (DANDI format): {experimenter_list}")

        # Extract sex information
        subject_sex = sub_info["sex"]
        # Convert database sex values to NWB format
        sex_mapping = {'Male': 'M', 'Female': 'F', 'Unknown': 'U'}
        subject_sex = sex_mapping.get(subject_sex, 'U')
        print(f"Subject sex: {subject_sex}")

        # Extract date of birth and convert to datetime
        if "dob" in sub_info and sub_info["dob"] is not None:
            dob = sub_info["dob"]
            # Convert date to datetime object (NWB requires datetime)
            if isinstance(dob, datetime):
                subject_date_of_birth = dob
            else:
                # If it's a date object, convert to datetime at midnight
                subject_date_of_birth = datetime.combine(dob, datetime.min.time())
            print(f"Subject date of birth: {subject_date_of_birth}")
        else:
            print("Subject date of birth: Not available in database")

    except Exception as e:
        print(f"Could not query experimenter info: {e}")
        import traceback
        traceback.print_exc()

    try:
        # Query session and recording information
        session_query_key = (acquisition.Session & session_key).fetch1('KEY')
        recording_keys = ((acquisition.Session * recording.Recording.BehaviorSession) & session_query_key).fetch(
            'recording_id', as_dict=True
        )

        if recording_keys:
            recording_id = recording_keys[0]['recording_id']
            print(f"Recording ID: {recording_id}")

            # Get compact sync_data and sampling rate from DataJoint
            sync_record = (ep.BehaviorSync & {'recording_id': recording_id}).fetch1()
            sync_data = sync_record['sync_data']
            nidq_rate = sync_record['nidq_sampling_rate']

            print(f"NIDQ sampling rate: {nidq_rate:.6f} Hz")
            print(f"Sync quality - Regular: {sync_record['regular_sync_status']}, "
                  f"Fixed: {sync_record['fixed_sync_status']}, "
                  f"Virmen: {sync_record['virmen_sync_status']}")

            # Choose appropriate sync method based on status
            if sync_record['regular_sync_status'] == 1 or sync_record['fixed_sync_status'] == 1:
                iteration_idx_vector = sync_data['iteration_idx_vector']
                sync_method = 'pulse-based'
            else:
                iteration_idx_vector = sync_data['iteration_idx_vector_from_virmen']
                sync_method = 'virmen-time-based'

            print(f"Using {sync_method} synchronization")
            print(f"Number of trials: {len(iteration_idx_vector)}")

            # Convert compact iteration indices to frame-level timestamps
            # All timestamps are relative to IMEC recording start (t=0)
            # Behavioral data starts AFTER IMEC starts (as per experimental protocol)
            frame_timestamps_list = []
            frame_count = 0
            first_nidq_sample = None

            for trial_num, iter_indices in enumerate(iteration_idx_vector):
                # Convert NIDQ sample indices to seconds (relative to IMEC start)
                trial_timestamps = iter_indices / nidq_rate

                # Track the first NIDQ sample for offset calculation
                if trial_num == 0:
                    first_nidq_sample = iter_indices[0]

                frame_timestamps_list.append(trial_timestamps)
                frame_count += len(trial_timestamps)

            # Concatenate all trial timestamps into single array
            # All timestamps are in IMEC time (t=0 at IMEC start)
            sync_timestamps = np.concatenate(frame_timestamps_list)

            # Calculate behavioral start offset for metadata
            behavioral_start_offset = first_nidq_sample / nidq_rate

            print(f"\nFrame-level timestamps created: {len(sync_timestamps)} frames")
            print(f"Time range: {sync_timestamps[0]:.3f}s to {sync_timestamps[-1]:.3f}s (relative to IMEC start)")
            print(f"Behavioral start offset: {behavioral_start_offset:.6f}s from IMEC start")
            print(f"Average frame rate: {len(sync_timestamps) / (sync_timestamps[-1] - sync_timestamps[0]):.2f} Hz")

        else:
            print("No recording ID found for this session")

    except Exception as e:
        print(f"Could not query sync timestamps: {e}")
        import traceback
        traceback.print_exc()
else:
    print("Skipping database queries - DB_AVAILABLE = False")


Experimenters (DANDI format): ['Yanar, Jorge']
Subject sex: F
Subject date of birth: 2023-08-29 00:00:00
Recording ID: 315
NIDQ sampling rate: 5000.060000 Hz
Sync quality - Regular: 1, Fixed: 0, Virmen: 1
Using pulse-based synchronization
Number of trials: 149

Frame-level timestamps created: 303515 frames
Time range: 25.350s to 3895.686s (relative to IMEC start)
Behavioral start offset: 25.350296s from IMEC start
Average frame rate: 78.42 Hz


In [ ]:
# Verify sync_timestamps match expected frame count from Virmen file
if sync_timestamps is not None:
    # Count expected frames from Virmen file
    mat_dict = virmen_interface._mat_dict
    if isinstance(mat_dict["log"]["block"], dict):
        epochs_raw = [mat_dict["log"]["block"]]
    else:
        epochs_raw = mat_dict["log"]["block"]

    trials_raw = [trial for epoch in epochs_raw for trial in epoch["trial"] if not np.isnan(trial["start"])]
    expected_frame_count = sum(len(trial["time"]) for trial in trials_raw)

    print(f"\n{'='*70}")
    print("TIMESTAMP VERIFICATION")
    print(f"{'='*70}")
    print(f"Expected frames from Virmen file: {expected_frame_count}")
    print(f"Sync timestamps created: {len(sync_timestamps)}")
    print(f"Match: {'✓ YES' if len(sync_timestamps) == expected_frame_count else '✗ NO'}")

    if len(sync_timestamps) != expected_frame_count:
        print(f"\n⚠ WARNING: Frame count mismatch!")
        print(f"  Difference: {len(sync_timestamps) - expected_frame_count} frames")
    print(f"{'='*70}\n")


TIMESTAMP VERIFICATION
Expected frames from Virmen file: 303515
Sync timestamps created: 303515
Match: ✓ YES



## 5b. Temporal Alignment Summary

Review the timing convention used throughout the NWB file:
- **All timestamps are relative to IMEC recording start (t=0)**
- Behavioral data starts after IMEC (delay documented as `behavioral_start_offset`)
- No timestamp conversion needed for analysis - everything is in IMEC time


In [ ]:
# Store behavioral start offset for metadata (already calculated in previous cell)
if 'behavioral_start_offset' in globals() and behavioral_start_offset is not None:
    print(f"\n{'='*70}")
    print("TEMPORAL ALIGNMENT SUMMARY")
    print(f"{'='*70}")
    print(f"Timing Convention: All timestamps relative to IMEC recording start (t=0)")
    print(f"\nSystem Start Times:")
    print(f"  - IMEC/NIDQ recording: t=0.000s (universal reference)")
    print(f"  - First behavioral frame: t={behavioral_start_offset:.6f}s")
    print(f"\nData Alignment:")
    print(f"  ✓ Kilosort spike times: In IMEC time (t ≥ 0)")
    print(f"  ✓ Behavioral timestamps: In IMEC time (t > 0, starts at ~{behavioral_start_offset:.1f}s)")
    print(f"  ✓ All data streams use common IMEC t=0 reference")
    print(f"  ✓ NO negative timestamps in any data stream")
    print(f"  ✓ NO conversion needed during analysis - already aligned!")
    print(f"\nCritical: Behavioral data ALWAYS starts AFTER IMEC recording (t > 0).")
    print(f"{'='*70}\n")

    # Verify sync_timestamps are positive
    if 'sync_timestamps' in globals() and sync_timestamps is not None:
        if np.all(sync_timestamps > 0):
            print(f"✓ Verified: All {len(sync_timestamps)} behavioral timestamps are positive")
        else:
            print(f"✗ WARNING: Some behavioral timestamps are ≤ 0!")
            print(f"  Min: {sync_timestamps.min():.6f}s, Max: {sync_timestamps.max():.6f}s")

    # Store for metadata
    metadata_behavioral_offset = behavioral_start_offset
else:
    metadata_behavioral_offset = None
    print("No sync data available - timestamps will use Virmen internal timing")


TEMPORAL ALIGNMENT SUMMARY
Timing Convention: All timestamps relative to IMEC recording start (t=0)

System Start Times:
  - IMEC/NIDQ recording: t=0.000s (universal reference)
  - First behavioral frame: t=25.350296s

Data Alignment:
  ✓ Kilosort spike times: In IMEC time (t ≥ 0)
  ✓ Behavioral timestamps: In IMEC time (t > 0, starts at ~25.4s)
  ✓ All data streams use common IMEC t=0 reference
  ✓ NO negative timestamps in any data stream
  ✓ NO conversion needed during analysis - already aligned!

Critical: Behavioral data ALWAYS starts AFTER IMEC recording (t > 0).

✓ Verified: All 303515 behavioral timestamps are positive


## 6. Initialize TowersNWBConverter

Create the converter with:
- **VirmenData** (required): Behavioral data  
- **KilosortProbe0, KilosortProbe1, ...** (optional): Spike sorting data from multiple probes
- **sync_timestamps**: Pre-computed synchronized timestamps from database

**Temporal Alignment Convention:**
- **IMEC recording start = t=0** (reference for all timestamps)
- Behavioral data starts AFTER IMEC (as per experimental protocol)
- All timestamps in NWB file are in IMEC time (no conversion needed for analysis)
- Kilosort spike times: Already in IMEC time
- Behavioral frame times: Adjusted to IMEC time (start at t=behavioral_start_offset)

**Note on multi-probe Kilosort:**
- The converter automatically detects and adds all Kilosort probes
- Each probe is added as a separate interface (KilosortProbe0, KilosortProbe1, etc.)
- All probes share the same IMEC recording start time (hardware synchronized)


## 5c. Discover Kilosort Probe Data

Automatically discover all Kilosort probes in the directory structure.

In [ ]:
import glob
from neuroconv.datainterfaces import KiloSortSortingInterface

# Discover all Kilosort probe directories
def discover_kilosort_probes(base_path):
    """
    Discover all Kilosort output directories for multiple probes.

    Looks for directories matching pattern: *_imec*/**/kilosort*_output
    For each probe, selects the highest job_id directory.

    Returns dict mapping probe names to kilosort output paths.
    """
    kilosort_probes = {}

    # Find all imec probe directories
    probe_dirs = sorted(base_path.glob("*_imec*"))

    for probe_dir in probe_dirs:
        probe_name = probe_dir.name  # e.g., "ya008_20240525_g0_imec0"
        probe_id = probe_name.split("_imec")[-1]  # e.g., "0"

        # Find all job directories
        job_dirs = sorted(probe_dir.glob("job_id_*"))

        if not job_dirs:
            print(f"  No job directories found in {probe_dir.name}")
            continue

        # Select highest job ID
        highest_job = sorted(job_dirs, key=lambda x: int(x.name.split("_")[-1]))[-1]

        # Find kilosort output directory
        kilosort_outputs = list(highest_job.glob("kilosort*_output"))

        if not kilosort_outputs:
            print(f"  No Kilosort output found in {highest_job.name}")
            continue

        kilosort_path = kilosort_outputs[0]
        interface_name = f"KilosortProbe{probe_id}"

        kilosort_probes[interface_name] = kilosort_path
        print(f"  Found {interface_name}: {kilosort_path}")

    return kilosort_probes

# Discover Kilosort probes
print("Discovering Kilosort probes...")
kilosort_probes = discover_kilosort_probes(kilosort_base)
print(f"\nFound {len(kilosort_probes)} Kilosort probe(s)")

Discovering Kilosort probes...
  Found KilosortProbe0: /Users/ct5868/code/testing_yanar/known_kilosort_structure/ya008_20240525_g0/ya008_20240525_g0_imec0/job_id_680/kilosort4_output
  Found KilosortProbe1: /Users/ct5868/code/testing_yanar/known_kilosort_structure/ya008_20240525_g0/ya008_20240525_g0_imec1/job_id_681/kilosort4_output

Found 2 Kilosort probe(s)


In [ ]:
# Define source data for converter with automatically discovered Kilosort probes
source_data = {
    "VirmenData": {
        "file_path": str(virmen_file_path)
    },
}

# Add all discovered Kilosort probes
for interface_name, kilosort_path in kilosort_probes.items():
    source_data[interface_name] = {
        "folder_path": str(kilosort_path),
        "keep_good_only": False  # Keep all units for analysis
    }
    print(f"Added {interface_name}: {kilosort_path.name}")

print(f"\nSource data interfaces: {list(source_data.keys())}")

# Initialize converter with sync timestamps
converter = TowersNWBConverter(
    source_data=source_data,
    sync_timestamps=sync_timestamps,  # Will be None if database not available
    ttl_source=None  # Could provide TTL file as fallback
)

print("\nConverter initialized successfully!")
print(f"Active interfaces: {list(converter.data_interface_objects.keys())}")

Added KilosortProbe0: kilosort4_output
Added KilosortProbe1: kilosort4_output

Source data interfaces: ['VirmenData', 'KilosortProbe0', 'KilosortProbe1']

Converter initialized successfully!
Active interfaces: ['VirmenData', 'KilosortProbe0', 'KilosortProbe1']


## 6b. Inspect Kilosort Data

Check the number of units in each probe.

In [ ]:
# Inspect Kilosort data from each probe
for interface_name, interface in converter.data_interface_objects.items():
    if interface_name.startswith("Kilosort"):
        # Get the SortingExtractor from the interface
        sorting = interface.sorting_extractor
        num_units = len(sorting.get_unit_ids())

        # Get some basic stats
        total_spikes = sum(len(sorting.get_unit_spike_train(unit_id)) for unit_id in sorting.get_unit_ids())

        print(f"\n{interface_name}:")
        print(f"  Number of units: {num_units}")
        print(f"  Total spikes: {total_spikes:,}")
        print(f"  Unit IDs (first 10): {sorting.get_unit_ids()[:10]}")

        # Show sampling frequency
        if hasattr(sorting, 'get_sampling_frequency'):
            print(f"  Sampling frequency: {sorting.get_sampling_frequency()} Hz")


KilosortProbe0:
  Number of units: 345
  Total spikes: 5,710,281
  Unit IDs (first 10): [0 1 2 3 4 5 6 7 8 9]
  Sampling frequency: 30000.0 Hz

KilosortProbe1:
  Number of units: 674
  Total spikes: 10,909,809
  Unit IDs (first 10): [0 1 2 3 4 5 6 7 8 9]
  Sampling frequency: 30000.0 Hz


## 7. Apply Synchronized Timestamps

Apply the pre-computed synchronized timestamps to all temporal interfaces.

In [ ]:
# Apply temporal alignment if sync timestamps are available
if sync_timestamps is not None:
    converter.temporally_align_data_interfaces()
    print("Temporal alignment applied successfully!")
else:
    print("No sync timestamps available - using original timestamps from Virmen file")

Temporal alignment applied successfully!


## 8. Create Metadata Dictionary

Prepare metadata including experimenter information and other database-derived fields. This will override the default metadata from the converter.

In [ ]:
# Get default metadata from converter
metadata = converter.get_metadata()

# Override with database-derived information
if experimenter_list:
    metadata["NWBFile"]["experimenter"] = experimenter_list
    print(f"Added experimenters to metadata: {experimenter_list}")

# Add subject sex if available
if subject_sex:
    metadata["Subject"]["sex"] = subject_sex
    print(f"Added subject sex to metadata: {subject_sex}")

# Add subject date of birth if available
if subject_date_of_birth is not None:
    metadata["Subject"]["date_of_birth"] = subject_date_of_birth
    print(f"Added subject date of birth to metadata: {subject_date_of_birth}")

# Add species
metadata["Subject"]["species"] = "Mus musculus"
print(f"Added species to metadata: Mus musculus")

# Add behavioral start offset for documentation purposes
if 'metadata_behavioral_offset' in globals() and metadata_behavioral_offset is not None:
    # Store in LabMetaData for accessibility
    if "LabMetaData" not in metadata:
        metadata["LabMetaData"] = {}
    metadata["LabMetaData"]["behavioral_start_offset"] = metadata_behavioral_offset
    print(f"\nAdded behavioral_start_offset to metadata: {metadata_behavioral_offset:.6f} seconds")
    print("  This documents when behavioral data started relative to IMEC t=0.")
    print("  All timestamps in NWB file are already in IMEC time (no conversion needed).")

# Display final metadata structure
print("\nFinal metadata structure:")
print(f"  Session ID: {metadata['NWBFile']['session_id']}")
print(f"  Institution: {metadata['NWBFile']['institution']}")
print(f"  Lab: {metadata['NWBFile']['lab']}")
print(f"  Subject ID: {metadata['Subject']['subject_id']}")
if subject_sex:
    print(f"  Subject sex: {metadata['Subject']['sex']}")
if subject_date_of_birth:
    print(f"  Subject date of birth: {metadata['Subject']['date_of_birth']}")
if experimenter_list:
    print(f"  Experimenters: {metadata['NWBFile']['experimenter']}")


Added experimenters to metadata: ['Yanar, Jorge']
Added subject sex to metadata: F
Added subject date of birth to metadata: 2023-08-29 00:00:00
Added species to metadata: Mus musculus

Added behavioral_start_offset to metadata: 25.350296 seconds
  This documents when behavioral data started relative to IMEC t=0.
  All timestamps in NWB file are already in IMEC time (no conversion needed).

Final metadata structure:
  Session ID: jorge_pwmv2_cohort1_185A-Rig1_jyanar_ya008_T_20240525_0
  Institution: Princeton
  Lab: Tank
  Subject ID: jyanar_ya008
  Subject sex: F
  Subject date of birth: 2023-08-29 00:00:00
  Experimenters: ['Yanar, Jorge']


## 9. Run Conversion

Execute the conversion process to create the NWB file with all metadata and synchronized timestamps.

In [ ]:
# Define output path
output_dir = Path("/Users/ct5868/code/tank-lab-to-nwb-clean/output")
output_dir.mkdir(exist_ok=True)
nwb_file_path = output_dir / f"{metadata['NWBFile']['session_id']}_synchronized.nwb"

# Run conversion
print(f"Converting to NWB file: {nwb_file_path}")
converter.run_conversion(
    nwbfile_path=str(nwb_file_path),
    metadata=metadata,
    overwrite=True
)

print(f"\n✓ Conversion complete! NWB file saved to: {nwb_file_path}")
print(f"  File size: {nwb_file_path.stat().st_size / 1024 / 1024:.2f} MB")

Converting to NWB file: /Users/ct5868/code/tank-lab-to-nwb-clean/output/jorge_pwmv2_cohort1_185A-Rig1_jyanar_ya008_T_20240525_0_synchronized.nwb


/Users/ct5868/code/tank-lab-to-nwb-clean/.venv/lib/python3.13/site-packages/pynwb/file.py:158: UserWarning: Date is missing timezone information. Updating to local timezone.
  args_to_set['date_of_birth'] = _add_missing_timezone(date_of_birth)


Adding behavioral timeseries data (Position, ViewAngle, Velocity, Collision)...
✓ Added 5 behavioral data interfaces to NWB file
  Data samples: 303515, Timestamps: 303515

✓ Conversion complete! NWB file saved to: /Users/ct5868/code/tank-lab-to-nwb-clean/output/jorge_pwmv2_cohort1_185A-Rig1_jyanar_ya008_T_20240525_0_synchronized.nwb
  File size: 63.01 MB


## 10. Verify NWB File Contents

Load and inspect the generated NWB file to verify that metadata, experimenter information, and temporal data were correctly populated.

In [ ]:
# Load the NWB file
with NWBHDF5IO(str(nwb_file_path), "r") as io:
    nwbfile = io.read()

    print("=" * 70)
    print("NWB FILE VERIFICATION")
    print("=" * 70)

    # General metadata
    print(f"\nSession Information:")
    print(f"  Session ID: {nwbfile.session_id}")
    print(f"  Session start: {nwbfile.session_start_time}")
    print(f"  Institution: {nwbfile.institution}")
    print(f"  Lab: {nwbfile.lab}")

    # Subject information
    print(f"\nSubject Information:")
    print(f"  Subject ID: {nwbfile.subject.subject_id}")
    if hasattr(nwbfile.subject, 'sex') and nwbfile.subject.sex:
        print(f"  Sex: {nwbfile.subject.sex}")
    if hasattr(nwbfile.subject, 'species') and nwbfile.subject.species:
        print(f"  Species: {nwbfile.subject.species}")

    # Experimenter information (database-derived)
    if hasattr(nwbfile, 'experimenter') and nwbfile.experimenter:
        print(f"  Experimenters: {nwbfile.experimenter}")
    else:
        print(f"  Experimenters: Not available (no database access)")

    # Lab metadata extension
    if hasattr(nwbfile, 'lab_meta_data') and 'LabMetaData' in nwbfile.lab_meta_data:
        lab_meta = nwbfile.lab_meta_data['LabMetaData']
        print(f"\nLab Metadata:")
        print(f"  Experiment: {lab_meta.experiment_name}")
        print(f"  Protocol: {lab_meta.protocol_name}")
        print(f"  Location: {lab_meta.location}")
        print(f"  Number of trials: {lab_meta.num_trials}")

    # Trials
    print(f"\nTrials:")
    print(f"  Number of trials: {len(nwbfile.trials)}")
    if len(nwbfile.trials) > 0:
        print(f"  Trial columns: {list(nwbfile.trials.colnames)}")

    # Behavioral data
    if 'behavior' in nwbfile.processing:
        print(f"\nBehavioral Data:")
        behavior_module = nwbfile.processing['behavior']
        print(f"  Available data: {list(behavior_module.data_interfaces.keys())}")

        if 'Position' in behavior_module.data_interfaces:
            position = behavior_module.data_interfaces['Position']['Position']
            print(f"  Position samples: {len(position.data)}")
            if sync_timestamps is not None:
                print(f"  ✓ Using synchronized timestamps from database")
            else:
                print(f"  Using original Virmen timestamps")

    # Units (Kilosort data)
    if nwbfile.units is not None and len(nwbfile.units) > 0:
        print(f"\nKilosort Units:")
        print(f"  Total units: {len(nwbfile.units)}")
        print(f"  Unit table columns: {list(nwbfile.units.colnames)}")

        # Count units per electrode group (probe)
        if 'electrode_group' in nwbfile.units.colnames:
            from collections import Counter
            probe_counts = Counter([eg.name for eg in nwbfile.units['electrode_group'][:]])
            for probe_name, count in sorted(probe_counts.items()):
                print(f"    {probe_name}: {count} units")

        # Show spike count stats
        total_spikes = sum(len(spikes) for spikes in nwbfile.units['spike_times'][:])
        print(f"  Total spikes: {total_spikes:,}")

    # Epochs
    if nwbfile.epochs is not None:
        print(f"\nEpochs:")
        print(f"  Number of epochs: {len(nwbfile.epochs)}")

    print("\n" + "=" * 70)
    print("VERIFICATION COMPLETE")
    print("=" * 70)

NWB FILE VERIFICATION

Session Information:
  Session ID: jorge_pwmv2_cohort1_185A-Rig1_jyanar_ya008_T_20240525_0
  Session start: 2024-05-25 12:29:55.539000-04:00
  Institution: Princeton
  Lab: Tank

Subject Information:
  Subject ID: jyanar_ya008
  Sex: F
  Species: Mus musculus
  Experimenters: ('Yanar, Jorge',)

Lab Metadata:
  Experiment: jessejorge_pwm_v3
  Protocol: pwm_jessejorge_v3
  Location: 185A-Rig1
  Number of trials: 149

Trials:
  Number of trials: 149
  Trial columns: ['start_time', 'stop_time', 'duration', 'trial_id', 'iterations', 'iCueEntry', 'iMemEntry', 'iTurnEntry', 'iArmEntry', 'iBlank', 'excessTravel', 'rewardScale', 'StartCycle', 'EndCycle', 'rule', 'trialNum', 'multibiasBeta', 'multibiasTau', 'pairNum', 'wallGuide', 'choice', 'trial_type', 'left_cue_presence', 'right_cue_presence', 'left_cue_onset', 'right_cue_onset', 'left_cue_offset', 'right_cue_offset', 'left_cue_position', 'right_cue_position', 'baseCycles', 'stimulusTable_pairNum', 'stimulusTable_prob',

## 11. Verify IMEC Time Reference

Verify that both Kilosort and behavioral data share the same IMEC t=0 reference, with behavioral data starting after t=0.

In [ ]:
# Verify IMEC time reference convention
# All timestamps (behavioral and spike) should be in IMEC time with t=0 at recording start

with NWBHDF5IO(str(nwb_file_path), "r") as io:
    nwbfile = io.read()

    print("=" * 70)
    print("IMEC TIME REFERENCE VERIFICATION")
    print("=" * 70)

    # Get the behavioral start offset from LabMetaData
    behavioral_start_offset = None
    if hasattr(nwbfile, 'lab_meta_data') and 'LabMetaData' in nwbfile.lab_meta_data:
        lab_meta = nwbfile.lab_meta_data['LabMetaData']
        if hasattr(lab_meta, 'behavioral_start_offset'):
            behavioral_start_offset = lab_meta.behavioral_start_offset
            print(f"\nBehavioral start offset: {behavioral_start_offset:.6f}s")
            print(f"  This documents when behavioral data started relative to IMEC t=0")

    # Get behavioral timestamps
    if 'behavior' in nwbfile.processing:
        position = nwbfile.processing['behavior']['Position']['SpatialSeries']
        behavioral_times = position.timestamps[:]

        print(f"\nBehavioral Timestamps (IMEC time reference):")
        print(f"  First frame: t={behavioral_times[0]:.6f}s")
        print(f"  Last frame: t={behavioral_times[-1]:.6f}s")
        print(f"  Total frames: {len(behavioral_times)}")

        # CRITICAL VERIFICATION: Behavioral times should be positive and > 0
        if np.all(behavioral_times > 0):
            print(f"  ✓ All behavioral timestamps are positive (> 0)")
        else:
            print(f"  ✗ ERROR: Some behavioral timestamps are ≤ 0!")
            print(f"    Min timestamp: {behavioral_times.min():.6f}s")

        # Verify behavioral starts after IMEC t=0
        if behavioral_times[0] > 0:
            print(f"  ✓ Behavioral data starts AFTER IMEC t=0 (as expected)")
        else:
            print(f"  ✗ ERROR: Behavioral data does not start after t=0!")

    # Get Kilosort spike times
    if nwbfile.units is not None and len(nwbfile.units) > 0:
        print(f"\nKilosort Spike Times (IMEC time reference):")
        print(f"  Total units: {len(nwbfile.units)}")

        # Get spike times from first unit
        spike_times = nwbfile.units['spike_times'][0]
        print(f"  Unit 0: {len(spike_times)} spikes")
        print(f"    First spike: t={spike_times[0]:.6f}s")
        print(f"    Last spike: t={spike_times[-1]:.6f}s")

        # CRITICAL VERIFICATION: Spike times should be ≥ 0
        if np.all(spike_times >= 0):
            print(f"  ✓ All spike times are non-negative (≥ 0)")
        else:
            print(f"  ✗ ERROR: Some spike times are negative!")
            print(f"    Min spike time: {spike_times.min():.6f}s")
            print(f"    Number of negative spikes: {np.sum(spike_times < 0)}")

        # Compare timing: behavioral should start after earliest spikes
        earliest_spike = np.min([nwbfile.units['spike_times'][i].min()
                                  for i in range(min(10, len(nwbfile.units)))])

        print(f"\n  Time Relationship:")
        print(f"    Earliest spike (first 10 units): t={earliest_spike:.6f}s")
        print(f"    First behavioral frame: t={behavioral_times[0]:.6f}s")

        if behavioral_times[0] > earliest_spike:
            print(f"  ✓ Behavioral starts AFTER earliest spikes (expected)")
        else:
            print(f"  ⚠ Behavioral starts before or at same time as spikes")

    print(f"\n{'=' * 70}")
    print("TIME REFERENCE SUMMARY")
    print(f"{'=' * 70}")
    print(f"✓ IMEC recording start = t=0 (universal reference)")
    print(f"✓ Kilosort spike times in IMEC time (no conversion needed)")
    print(f"✓ Behavioral timestamps in IMEC time (no conversion needed)")
    print(f"✓ Both data streams already aligned to same reference")
    print(f"✓ NO negative timestamps (all data after IMEC trigger)")
    print(f"{'=' * 70}\n")

IMEC TIME REFERENCE VERIFICATION

Behavioral Timestamps (IMEC time reference):
  First frame: t=25.350296s
  Last frame: t=3895.686252s
  Total frames: 303515
  ✓ All behavioral timestamps are positive (> 0)
  ✓ Behavioral data starts AFTER IMEC t=0 (as expected)

Kilosort Spike Times (IMEC time reference):
  Total units: 674
  Unit 0: 6531 spikes
    First spike: t=0.059767s
    Last spike: t=3518.910533s
  ✓ All spike times are non-negative (≥ 0)

  Time Relationship:
    Earliest spike (first 10 units): t=0.006767s
    First behavioral frame: t=25.350296s
  ✓ Behavioral starts AFTER earliest spikes (expected)

TIME REFERENCE SUMMARY
✓ IMEC recording start = t=0 (universal reference)
✓ Kilosort spike times in IMEC time (no conversion needed)
✓ Behavioral timestamps in IMEC time (no conversion needed)
✓ Both data streams already aligned to same reference
✓ NO negative timestamps (all data after IMEC trigger)



## 11. Debug: Check Raw Data

Let's examine the raw Virmen data to see how many epochs/blocks actually exist in the file.

In [ ]:
# Check the raw virmen data structure
from tank_lab_to_nwb.utils import convert_mat_file_to_dict

mat_dict = convert_mat_file_to_dict(str(virmen_file_path))

# Check if block is dict or list
if isinstance(mat_dict["log"]["block"], dict):
    print(f"Block is a dict (single epoch)")
    epochs_raw = [mat_dict["log"]["block"]]
else:
    print(f"Block is a list/array with {len(mat_dict['log']['block'])} epochs")
    epochs_raw = mat_dict["log"]["block"]

print(f"\nNumber of epochs: {len(epochs_raw)}")
for i, epoch in enumerate(epochs_raw):
    num_trials = sum(1 for trial in epoch["trial"] if not np.isnan(trial["start"]))
    print(f"  Epoch {i+1}: {num_trials} trials, mazeID={epoch['mazeID']}, duration={epoch['duration']:.2f}s")

Block is a list/array with 3 epochs

Number of epochs: 3
  Epoch 1: 10 trials, mazeID=8, duration=142.11s
  Epoch 2: 137 trials, mazeID=9, duration=3596.22s
  Epoch 3: 2 trials, mazeID=8, duration=132.10s


In [ ]:
# Check if the VirmenDataInterface itself has the method
print(f"VirmenDataInterface has add_to_nwbfile: {hasattr(virmen_interface, 'add_to_nwbfile')}")

# Check if the converter actually calls add_to_nwbfile
print(f"\nConverter data interfaces: {list(converter.data_interface_objects.keys())}")

# Let's check what conversion options are being used
print(f"\nChecking conversion behavior...")

# Test if the issue is in how neuroconv calls add_to_nwbfile
import inspect
print(f"\nadd_to_nwbfile signature: {inspect.signature(virmen_interface.add_to_nwbfile)}")

VirmenDataInterface has add_to_nwbfile: True

Converter data interfaces: ['VirmenData', 'KilosortProbe0', 'KilosortProbe1']

Checking conversion behavior...

add_to_nwbfile signature: (nwbfile: pynwb.file.NWBFile, metadata: dict)


In [ ]:
# Reload the modules to pick up the code changes
import importlib
import tank_lab_to_nwb.convert_towers_task.virmenbehaviordatainterface
import tank_lab_to_nwb.convert_towers_task.towersnwbconverter

importlib.reload(tank_lab_to_nwb.convert_towers_task.virmenbehaviordatainterface)
importlib.reload(tank_lab_to_nwb.convert_towers_task.towersnwbconverter)

from tank_lab_to_nwb.convert_towers_task.towersnwbconverter import TowersNWBConverter
from tank_lab_to_nwb.convert_towers_task.virmenbehaviordatainterface import VirmenDataInterface

print("Modules reloaded!")

Modules reloaded!


In [ ]:
# Re-initialize converter with sync timestamps
converter = TowersNWBConverter(
    source_data=source_data,
    sync_timestamps=sync_timestamps,
    ttl_source=None
)

# Apply temporal alignment
converter.temporally_align_data_interfaces()

# Re-run conversion
print(f"Converting to NWB file: {nwb_file_path}")
converter.run_conversion(
    nwbfile_path=str(nwb_file_path),
    metadata=metadata,
    overwrite=True
)

print(f"\n✓ Conversion complete! NWB file saved to: {nwb_file_path}")
print(f"  File size: {nwb_file_path.stat().st_size / 1024 / 1024:.2f} MB")

Converting to NWB file: /Users/ct5868/code/tank-lab-to-nwb-clean/output/jorge_pwmv2_cohort1_185A-Rig1_jyanar_ya008_T_20240525_0_synchronized.nwb


/Users/ct5868/code/tank-lab-to-nwb-clean/.venv/lib/python3.13/site-packages/pynwb/file.py:158: UserWarning: Date is missing timezone information. Updating to local timezone.
  args_to_set['date_of_birth'] = _add_missing_timezone(date_of_birth)


Adding behavioral timeseries data (Position, ViewAngle, Velocity, Collision)...
✓ Added 5 behavioral data interfaces to NWB file
  Data samples: 303515, Timestamps: 303515

✓ Conversion complete! NWB file saved to: /Users/ct5868/code/tank-lab-to-nwb-clean/output/jorge_pwmv2_cohort1_185A-Rig1_jyanar_ya008_T_20240525_0_synchronized.nwb
  File size: 63.00 MB


In [ ]:
# Verify the fixed NWB file
with NWBHDF5IO(str(nwb_file_path), "r") as io:
    nwbfile = io.read()

    print("=" * 70)
    print("UPDATED NWB FILE VERIFICATION")
    print("=" * 70)

    # Behavioral data
    if 'behavior' in nwbfile.processing:
        behavior_module = nwbfile.processing['behavior']
        print(f"\nBehavioral Data Interfaces:")
        print(f"  Available data: {list(behavior_module.data_interfaces.keys())}")

        if 'Position' in behavior_module.data_interfaces:
            position = behavior_module.data_interfaces['Position']['SpatialSeries']
            print(f"  Position samples: {len(position.data)}")
            print(f"  Position data shape: {position.data.shape}")

        if 'Velocity' in behavior_module.data_interfaces:
            velocity = behavior_module.data_interfaces['Velocity']
            print(f"  Velocity samples: {len(velocity.data)}")

        if 'ViewAngle' in behavior_module.data_interfaces:
            view_angle = behavior_module.data_interfaces['ViewAngle']['SpatialSeries']
            print(f"  ViewAngle samples: {len(view_angle.data)}")

    print(f"\n  File size: {nwb_file_path.stat().st_size / 1024 / 1024:.2f} MB")
    print("=" * 70)

UPDATED NWB FILE VERIFICATION

Behavioral Data Interfaces:
  Available data: ['Collision', 'Position', 'Time', 'Velocity', 'ViewAngle']
  Position samples: 303515
  Position data shape: (303515, 2)
  Velocity samples: 303515
  ViewAngle samples: 303515

  File size: 63.00 MB


In [ ]:
# Verify timing convention
print("="*70)
print("TIMING VERIFICATION")
print("="*70)
print(f"First 5 behavioral timestamps (in IMEC time):")
print(f"  {sync_timestamps[:5]}")
print(f"\nLast 5 behavioral timestamps (in IMEC time):")
print(f"  {sync_timestamps[-5:]}")
# print(f"\nBehavioral start offset: {behavioral_start_offset:.6f}s")
# print(f"First timestamp matches offset: {np.isclose(sync_timestamps[0], behavioral_start_offset)}")
print(f"\n✓ All timestamps are in IMEC time (t=0 at IMEC start)")
print("="*70)


TIMING VERIFICATION
First 5 behavioral timestamps (in IMEC time):
  [25.3502958  25.36989556 25.39449527 25.42409491 25.45309456]

Last 5 behavioral timestamps (in IMEC time):
  [3895.37925545 3895.39025532 3895.40185518 3895.41425503 3895.68625176]

✓ All timestamps are in IMEC time (t=0 at IMEC start)


In [ ]:
# Verify timestamps in NWB file
with NWBHDF5IO(str(nwb_file_path), "r") as io:
    nwbfile = io.read()

    print("="*70)
    print("NWB FILE TIMESTAMP VERIFICATION")
    print("="*70)

    # Check behavioral timestamps
    if 'behavior' in nwbfile.processing:
        position = nwbfile.processing['behavior']['Position']['SpatialSeries']
        behavioral_times = position.timestamps[:]

        print(f"\nBehavioral Timestamps (in IMEC time):")
        print(f"  First 5: {behavioral_times[:5]}")
        print(f"  Last 5: {behavioral_times[-5:]}")
        print(f"  Time range: {behavioral_times[0]:.3f}s to {behavioral_times[-1]:.3f}s")

    # Check Kilosort timestamps
    if nwbfile.units is not None and len(nwbfile.units) > 0:
        print(f"\nKilosort Units:")
        print(f"  Total units: {len(nwbfile.units)}")

        # Show spike times from first unit
        first_unit_spikes = nwbfile.units['spike_times'][0][:10]
        print(f"  First 10 spikes from unit 0: {first_unit_spikes}")
        print(f"  (All spike times in IMEC time)")

    # Check metadata
    if hasattr(nwbfile, 'lab_meta_data') and 'LabMetaData' in nwbfile.lab_meta_data:
        lab_meta = nwbfile.lab_meta_data['LabMetaData']
        if hasattr(lab_meta, 'behavioral_start_offset'):
            print(f"\nBehavioral start offset from metadata: {lab_meta.behavioral_start_offset:.6f}s")

    print(f"\n✓ All timestamps are in IMEC time (t=0 at IMEC start)")
    print(f"✓ Behavioral frames start at t≈{behavioral_times[0]:.1f}s")
    print(f"✓ Kilosort spikes already in same time reference")
    print("="*70)


NWB FILE TIMESTAMP VERIFICATION

Behavioral Timestamps (in IMEC time):
  First 5: [25.3502958  25.36989556 25.39449527 25.42409491 25.45309456]
  Last 5: [3895.37925545 3895.39025532 3895.40185518 3895.41425503 3895.68625176]
  Time range: 25.350s to 3895.686s

Kilosort Units:
  Total units: 674
  First 10 spikes from unit 0: [0.05976667 0.06443333 0.07373333 0.1021     0.13993333 0.1517
 0.19653333 0.20686667 0.27526667 0.31946667]
  (All spike times in IMEC time)

✓ All timestamps are in IMEC time (t=0 at IMEC start)
✓ Behavioral frames start at t≈25.4s
✓ Kilosort spikes already in same time reference
